# TakeOff.ai Mask R-CNN GPU preflight

This notebook performs one bounded train-step smoke test against three hash-pinned real samples from the frozen Swiss Dwellings adapter. It does **not** start model training.

Expected inputs: private Kaggle datasets `takeoff-maskrcnn-preflight-v2` and `takeoff-task6-preflight-source-v2`.

## 1. Verify the Kaggle runtime

In [ ]:
import platform, subprocess, sys
print('Python:', platform.python_version())
assert sys.version_info[:2] == (3, 12), 'This package requires the current Kaggle Python 3.12 runtime'
subprocess.run(['nvidia-smi'], check=True)

## 2. Install the pinned CUDA framework

Internet must be enabled for this cell.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--upgrade', 'torch==2.5.1', 'torchvision==0.20.1', '--index-url', 'https://download.pytorch.org/whl/cu121'], check=True)

## 3. Locate the two attached private Kaggle datasets

In [ ]:
from pathlib import Path
import sys, zipfile
input_root = Path('/kaggle/input')
bundle_candidates = [path for path in input_root.rglob('preflight-bundle-manifest.json') if 'takeoff-maskrcnn-preflight-v2' in path.as_posix().lower()]
source_candidates = [path for path in input_root.rglob('maskrcnn_kaggle_preflight.py') if 'takeoff-task6-preflight-source-v2' in path.as_posix().lower()]
if not bundle_candidates:
    bundle_archives = list(input_root.rglob('takeoff-maskrcnn-preflight-v2.zip'))
    assert len(bundle_archives) == 1, f'Expected one bundle archive, found {bundle_archives}'
    bundle_extract = Path('/kaggle/working/preflight-bundle-v2')
    with zipfile.ZipFile(bundle_archives[0]) as archive:
        archive.extractall(bundle_extract)
    bundle_candidates = list(bundle_extract.rglob('preflight-bundle-manifest.json'))
if not source_candidates:
    source_archives = list(input_root.rglob('takeoff-task6-preflight-source-v2.zip'))
    assert len(source_archives) == 1, f'Expected one source archive, found {source_archives}'
    source_extract = Path('/kaggle/working/preflight-source-v2')
    with zipfile.ZipFile(source_archives[0]) as archive:
        archive.extractall(source_extract)
    source_candidates = list(source_extract.rglob('maskrcnn_kaggle_preflight.py'))
assert len(bundle_candidates) == 1, f'Expected one preflight bundle, found {bundle_candidates}'
assert len(source_candidates) == 1, f'Expected one source package, found {source_candidates}'
bundle_dir = bundle_candidates[0].parent
backend_dir = next(path for path in source_candidates[0].parents if (path / 'ml').is_dir()).resolve()
assert 'takeoff-task6-preflight-source-v1' not in backend_dir.as_posix().lower(), backend_dir
sys.path[:] = [entry for entry in sys.path if 'takeoff-task6-preflight-source-v1' not in str(entry).lower()]
sys.path.insert(0, str(backend_dir))
print('Bundle:', bundle_dir)
print('V2 backend source:', backend_dir)

## 4. Run the single-step preflight

In [ ]:
import importlib, json, sys
sys.path[:] = [entry for entry in sys.path if 'takeoff-task6-preflight-source-v1' not in str(entry).lower() and Path(entry or '.').resolve() != backend_dir]
sys.path.insert(0, str(backend_dir))
for module_name in [name for name in tuple(sys.modules) if name == 'ml' or name.startswith('ml.')]:
    del sys.modules[module_name]
importlib.invalidate_caches()
preflight_module = importlib.import_module('ml.training.maskrcnn_kaggle_preflight')
module_path = Path(preflight_module.__file__).resolve()
assert module_path.is_relative_to(backend_dir), f'Wrong preflight source imported: {module_path}; expected under {backend_dir}'
print('Imported V2 preflight module:', module_path)
report = preflight_module.run_gpu_preflight(bundle_dir)
report_path = Path('/kaggle/working/maskrcnn-gpu-preflight-report.json')
report_path.write_text(json.dumps(report, indent=2, sort_keys=True) + '\n')
print(json.dumps(report, indent=2, sort_keys=True))

## 5. Assert every required PASS criterion

In [ ]:
assert report['dataset_samples'] == 3
assert report['hole_instances_checked'] > 0
assert report['intentional_negative_verified'] is True
assert report['backward'] is True
assert report['optimizer_step'] is True
assert report['inference_contract']['compatible'] is True
assert report['peak_reserved_gib'] < report['vram_gib']
assert report['training_started'] is False
print('MASK R-CNN GPU PREFLIGHT: PASS')

## Next step

Download `maskrcnn-gpu-preflight-report.json` and review it. Do not enable any epoch-based training cell; this notebook intentionally contains none.